# MODIS Feature Extraction (MOD13Q1)

Extracts MODIS features described in the dataset overview. By default, overlapping reflectance bands with Landsat (red/NIR/blue) are excluded.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import pystac_client
import planetary_computer as pc
from odc.stac import stac_load

from datetime import date
from tqdm import tqdm
import os
import time
import random
import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
os.environ["CURL_CA_BUNDLE"] = certifi.where()

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Exclude overlap with Landsat reflectance bands by default
INCLUDE_OVERLAP = False

tqdm.pandas()

In [ ]:
# Optional: install dependencies if missing
# !pip install numpy pandas odc-stac pystac-client planetary-computer tqdm certifi

In [ ]:
CATALOG = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

BASE_FEATURES = [
    "ndvi",
    "evi",
    "mir",
    "vi_quality",
    "pixel_reliability",
    "sun_zenith",
    "view_zenith",
    "rel_azimuth",
    "day_of_year",
]

OVERLAP_FEATURES = ["red", "nir", "blue"]

FEATURES = BASE_FEATURES + (OVERLAP_FEATURES if INCLUDE_OVERLAP else [])

ASSET_CANDIDATES = {
    "ndvi": ["ndvi"],
    "evi": ["evi"],
    "red": ["red", "sur_refl_b01", "b01"],
    "nir": ["nir", "sur_refl_b02", "b02"],
    "blue": ["blue", "sur_refl_b03", "b03"],
    "mir": ["mir", "swir", "sur_refl_b07", "b07"],
    "vi_quality": ["vi_quality", "vi quality", "quality"],
    "pixel_reliability": ["pixel_reliability", "pixel reliability", "reliability"],
    "sun_zenith": ["sun_zenith", "solar_zenith", "sun zenith"],
    "view_zenith": ["view_zenith", "view zenith"],
    "rel_azimuth": ["relative_azimuth", "rel_azimuth", "azimuth"],
    "day_of_year": ["day_of_year", "day of year", "doy"],
}


def _match_asset_key(item, candidates):
    asset_keys = list(item.assets.keys())
    for cand in candidates:
        if cand in asset_keys:
            return cand
        cand_lower = cand.lower()
        for key in asset_keys:
            if cand_lower in key.lower():
                return key
    return None


def _build_asset_map(item):
    asset_map = {}
    for feature in FEATURES:
        asset_map[feature] = _match_asset_key(item, ASSET_CANDIDATES[feature])
    return asset_map


def compute_modis_values(row, max_retries: int = 5, base_sleep_s: float = 1.0):
    lat = row['Latitude']
    lon = row['Longitude']
    sample_date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')

    default_return = pd.Series({feature: np.nan for feature in FEATURES})

    if pd.isna(sample_date):
        return default_return

    bbox_size = 0.0025
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2,
    ]

    search = CATALOG.search(
        collections=["modis-13Q1-061"],
        bbox=bbox,
        datetime="2011-01-01/2015-12-31",
    )

    items = search.item_collection()
    if not items:
        return default_return

    sample_date_utc = sample_date.tz_localize("UTC") if sample_date.tzinfo is None else sample_date.tz_convert("UTC")

    last_err = None
    for attempt in range(max_retries):
        try:
            items_sorted = sorted(
                items,
                key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc),
            )
            selected_item = pc.sign(items_sorted[0])

            asset_map = _build_asset_map(selected_item)
            available_bands = [key for key in asset_map.values() if key]
            if not available_bands:
                return default_return

            data = stac_load([selected_item], bands=available_bands, bbox=bbox).isel(time=0)

            def safe_median(asset_key: str):
                if not asset_key or asset_key not in data:
                    return np.nan
                try:
                    band_data = data[asset_key].astype("float")
                    median_val = float(band_data.median(skipna=True).values)
                    return median_val if median_val != 0 else np.nan
                except Exception:
                    return np.nan

            return pd.Series({
                feature: safe_median(asset_map[feature])
                for feature in FEATURES
            })

        except Exception as e:
            last_err = e
            sleep_s = base_sleep_s * (2 ** attempt) + random.random() * 0.25
            time.sleep(sleep_s)

    return default_return


In [ ]:
# Load data
Water_Quality_df = pd.read_csv(os.path.join(PROJECT_ROOT, 'water_quality_training_dataset.csv'))
Validation_df = pd.read_csv(os.path.join(PROJECT_ROOT, 'submission_template.csv'))

print(f"Training rows: {len(Water_Quality_df)}")
print(f"Validation rows: {len(Validation_df)}")

In [ ]:
chunk_size = 200

train_features_path = os.path.join(PROJECT_ROOT, 'New Datasets', 'modis_features_training_allvars.csv')
val_features_path = os.path.join(PROJECT_ROOT, 'New Datasets', 'modis_features_validation_allvars.csv')

expected_cols = ['Latitude', 'Longitude', 'Sample Date'] + FEATURES


def count_rows_in_csv(path: str) -> int:
    with open(path, 'r', encoding='utf-8') as f:
        return max(sum(1 for _ in f) - 1, 0)


def extract_chunked(input_df: pd.DataFrame, output_path: str, label: str):
    start_idx = 0
    if os.path.exists(output_path):
        existing_header = pd.read_csv(output_path, nrows=0).columns.tolist()
        if existing_header != expected_cols:
            raise ValueError(
                f"Existing file has different columns.\n"
                f"Expected: {expected_cols}\n"
                f"Found:    {existing_header}\n"
                f"Fix: delete/rename the existing file or update expected_cols."
            )
        start_idx = count_rows_in_csv(output_path)

    print(f"🚀 Running MODIS extraction for {label} (chunked)...")
    print(f"Total rows: {len(input_df)}")
    print(f"Output file: {output_path}")
    print(f"Chunk size: {chunk_size}")
    print(f"Resuming from row index: {start_idx}")

    for chunk_start in range(start_idx, len(input_df), chunk_size):
        chunk_end = min(chunk_start + chunk_size, len(input_df))
        chunk_df = input_df.iloc[chunk_start:chunk_end].copy()

        print(f"\nProcessing rows {chunk_start}..{chunk_end-1} ({len(chunk_df)} rows)")
        try:
            chunk_feats = chunk_df.progress_apply(compute_modis_values, axis=1)

            chunk_feats['Latitude'] = chunk_df['Latitude'].values
            chunk_feats['Longitude'] = chunk_df['Longitude'].values
            chunk_feats['Sample Date'] = chunk_df['Sample Date'].values

            chunk_out = chunk_feats[expected_cols]
            write_header = (not os.path.exists(output_path)) or (count_rows_in_csv(output_path) == 0)
            chunk_out.to_csv(output_path, mode='a', header=write_header, index=False)

            time.sleep(0.5)

        except Exception as e:
            print(f"\n❌ Chunk failed at rows {chunk_start}..{chunk_end-1}: {e}")
            print("You can rerun this cell to resume from the last completed chunk.")
            break


extract_chunked(Water_Quality_df, train_features_path, "training")
extract_chunked(Validation_df, val_features_path, "validation")

if os.path.exists(train_features_path):
    display(pd.read_csv(train_features_path).head())
if os.path.exists(val_features_path):
    display(pd.read_csv(val_features_path).head())